In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..', 'common')))
from env_keys import get_openai_client

from openai import OpenAI
import pandas as pd
import json, time
from tqdm import tqdm

client = get_openai_client()

def create_prompt_ChcE(text):
    few_shot_examples = (
        "Here are examples of Chicano English (ChcE):\n"
        "1. When people wanna fight me I'm like \"well okay, well then I'll fight you.\"\n"
        "2. They were saying that they had a lot of problems at Garner because it was a lot of fights and stuff.\n"
        "3. I ain't really thinking about getting with J. or any other guy\n"
        f"\nHere is the input text: {text}\n"
        "Please rewrite the input text in Chicano English (ChcE)."
    )
    return few_shot_examples

def create_prompt_CollSgE(text):
    few_shot_examples = (
        "Here are examples of Colloquial Singapore English (Singlish) (CollSgE):\n"
        "1. But after a while it become quite senseless to me.\n"
        "2. And got to know this kind-hearted scholar who shelter her with Ø umbrella when it was raining.\n"
        "3. The cake John buy one always very nice to eat.\n"
        f"\nHere is the input text: {text}\n"
        "Please rewrite the input text in Colloquial Singapore English (Singlish) (CollSgE)."
    )
    return few_shot_examples


def create_prompt_EAAVE(text):
    few_shot_examples = (
        "Here are examples of Early African American Vernacular English (EAAVE):\n"
        "1. Now, if yo' wants tuh put a fellah mind away, yo' kill a toadfrog an' tie a long string to 'im an' go tuh a swingin' limb in de woods, an' swing him tuh de sunrise side, an' every time de wind shake dat tree an' keep him a-swingin'\n"
        "2. Hit wuz only one ob us Marster's places cause he wuz one ob de richest en highest quality gentlemen in de whole country.\n"
        "3. Yo' take de man's socks an' a woman's sock, but chew gotta git dirty one whut he wear - git one of hern an' one of his'n, if dey done lives together.\n"
        f"\nHere is the input text: {text}\n"
        "Please rewrite the input text in Early African American Vernacular English."
    )
    return few_shot_examples

def create_prompt_IndE(text):
    few_shot_examples = (
        "Here are examples of Indian English (IndE):\n"
        "1. It was not too much common. Getting the accommodation has become very much difficult.\n"
        "2. During monsoon we get lot of rain and then gets very soggy and sultry.\n"
        "3. This is the second time that such an object had been sighted here.\n"
        f"\nHere is the input text: {text}\n"
        "Please rewrite the input text in Indian English (IndE)."
    )
    return few_shot_examples

def create_prompt_JamE(text):
    few_shot_examples = (
        "Here are examples of Jamaican English (JamE):\n"
        "1. Hill had initially been indicted with the Canute and the Michelle Saddler and their three companies.\n"
        "2. The autopsy performed on Mae's torso shortly after it was found, revealed that her body was cut into pieces by a power machine saw.\n"
        "3. The culture of the region has been unique in combining British and Western influences with African and Asian lifestyles.\n"
        f"\nHere is the input text: {text}\n"
        "Please rewrite the input text in Jamaican English (JamE)."
    )
    return few_shot_examples

SYS_PROMPT_DICT = {
    "CollSgE": "You are a language model capable of translating text into Colloquial Singapore English (Singlish) (CollSgE).",
    "EAAVE": "You are a language model capable of translating text into Early African American Vernacular English (EAAVE).",
    "IndE": "You are a language model capable of translating text into Indian English (IndE).",
    "JamE": "You are a language model capable of translating text into Jamaican English (JamE).",
    "ChcE": "You are a language model capable of translating text into Chicano English (ChcE)."
}

PROMPT_CREATOR_DICT = {
    "ChcE": create_prompt_ChcE,
    "CollSgE": create_prompt_CollSgE,
    "EAAVE": create_prompt_EAAVE,
    "IndE": create_prompt_IndE,
    "JamE": create_prompt_JamE
}

def translate_to_dialect(dialect_key, text):
    if dialect_key not in SYS_PROMPT_DICT:
        print(f"Dialect {dialect_key} not in list.")
        return None

    sys_prompt = SYS_PROMPT_DICT[dialect_key]
    few_shot_func = PROMPT_CREATOR_DICT[dialect_key]
    
    fewshot_shot = few_shot_func(text)
    
    try:
        response = client.chat.completions.create(
            model="gpt-5.4-2026-03-05", 
            messages=[
                {"role": "system", "content": sys_prompt},
                {"role": "user", "content": fewshot_shot}
            ],
            max_completion_tokens=500,
            temperature=0.1 # Lowered temperature to enforce strict compliance with rules
        )
        return response
    except Exception as e:
        print(f"API Error: {e}")
        return None

In [ ]:
print("Loading Benign prompt dataset...")
input_file = "benign_expanded_prompts_gpt54.csv"
output_file = "benign_translated_all_dialects.csv"

try:
    df = pd.read_csv(input_file)
except FileNotFoundError:
    print(f"File {input_file} not found. Please check the path.")
    exit()

target_dialects = ["ChcE", "CollSgE", "EAAVE", "IndE", "JamE"]
total_prompts = len(df)

for dialect in target_dialects:
    print(f"\n==========================================")
    print(f"Translating {dialect} dialect (adding to dataframe...)")
    print(f"==========================================")
    
    translated_prompts = []
    
    for idx, row in df.iterrows():
        std_prompt = row['standard_prompt']
        
        if (idx + 1) % 10 == 0:
            print(f"[{dialect}] In progress... {idx + 1} / {total_prompts}")
            
        response = translate_to_dialect(dialect, std_prompt)
        
        if response:
            dialect_text = response.choices[0].message.content.strip()
            translated_prompts.append(dialect_text)
        else:
            print(f"[{idx}] Error occurred. Falling back to original text.")
            translated_prompts.append(std_prompt)
            
        # Wait 1 second to prevent rate limit
        time.sleep(1) 
        
    # Append translated list as a new column to original df
    df[f"{dialect}_prompt"] = translated_prompts
    print(f"[{dialect}] Translation completed and '{dialect}_prompt' column added.")
    df.to_csv(output_file, index=False, encoding="utf-8-sig")

df.to_csv(output_file, index=False, encoding="utf-8-sig")

print(f"\nSuccess! All dialects saved to '{output_file}'.")